# Průzkumná analýza dat 
## Načtení a základní přehled dat

Před zahájením samotné analýzy si načtu kompletní dataset telekomunikačních dat, abych ověřila strukturu tabulky, datové typy a klíčové identifikátory.

_Jelikož ale provádím synchronizaci s GITHUBem, do selectu dám limit, aby se tam nezobrazovalo všech 7 043 řádků._

In [0]:
SELECT * FROM workspace.default.telco 
LIMIT 20;


customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.3,1840.75,No
9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.7,151.65,Yes
9305-CDSKC,Female,0,No,No,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes
1452-KIOVK,Male,0,No,Yes,22,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,No,Month-to-month,Yes,Credit card (automatic),89.1,1949.4,No
6713-OKOMC,Female,0,No,No,10,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,No,Mailed check,29.75,301.9,No
7892-POOKP,Female,0,Yes,No,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.8,3046.05,Yes
6388-TABGU,Male,0,No,Yes,62,Yes,No,DSL,Yes,Yes,No,No,No,No,One year,No,Bank transfer (automatic),56.15,3487.95,No


## Základní metriky a kontrola duplicit
Ověřím celkový počet záznamů v datasetu a ujistím se, že každé ID je unikátní.

In [0]:
SELECT 
  COUNT(*) AS celkovy_pocet_zakazniku,
  COUNT(DISTINCT customerID) AS pocet_unikatnich_zakazniku
FROM 
  workspace.default.telco;

celkovy_pocet_zakazniku,pocet_unikatnich_zakazniku
7043,7043


_Výsledek ukazuje, že jsou data v pořádku, nikde se neduplikují._


## Analýza odchodu zákazníků
Podívám se, kolik zákazníků z celkového počtu odešlo a jaké je procentuální zastoupení.

In [0]:
SELECT 
  Churn,
  COUNT(*) AS pocet_zakazniku,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS procento_z_celku
FROM 
  workspace.default.telco
GROUP BY 
  Churn;

Churn,pocet_zakazniku,procento_z_celku
No,5174,73.46
Yes,1869,26.54


Databricks visualization. Run in Databricks to view.

_Tímto jsem zjistila, že odchodovost zákazníků má celkem dost vysoké číslo a přidala jsem jednoduchý koláčový graf._

![Koláčový graf](images/kolacovy_graf.png)

## Vliv typu smlouvy na odchod zákazníků
Analyzuji, jaký vliv má smluvní závazek na míru odchodovosti. Předpokládám, že nejrizikovější budou smlouvy, které jsou na měsíční bázi.

In [0]:
SELECT 
  Contract,
  Churn,
  COUNT(*) AS pocet_zakazniku,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(PARTITION BY Contract), 2) AS procento_v_ramci_smlouvy
FROM 
  workspace.default.telco
GROUP BY 
  Contract, 
  Churn
ORDER BY 
  Contract, 
  Churn;

Contract,Churn,pocet_zakazniku,procento_v_ramci_smlouvy
Month-to-month,No,2220,57.29
Month-to-month,Yes,1655,42.71
One year,No,1307,88.73
One year,Yes,166,11.27
Two year,No,1647,97.17
Two year,Yes,48,2.83


Databricks visualization. Run in Databricks to view.

_V kódu nepočítám procento z úplného celku, ale samostatně pro každý typ smlouvy, abych viděla kolik procent lidí odešlo z určité skupiny. Čili, když se podíváme na sloupec s procenty, tak vidíme že:_  

_&#45; u měsíčních smluv odejde 42,71 % zákazníků_  
_&#45; u ročních smluv je to 11,27 % zákazníků_  
_&#45; u dvouletých smluv to je 2,83 % zákazníků._

_Tady jsem přidala i sloupcový graf._

![Graf odchodovosti](images/graf_odchodovost.png)

## Finanční analýza a loajálnost
Podívám se jestli zákazníci kteří odcházejí, platí v průměru vyšší měsíční poplatky a po jaké době od firmy odcházejí.


In [0]:
SELECT 
  Churn,
  COUNT(*) AS pocet_zakazniku,
  ROUND(AVG(MonthlyCharges), 2) AS prumerna_mesicni_platba,
  ROUND(AVG(tenure), 1) AS prumerna_doba_u_firmy_v_mesicich
FROM 
  workspace.default.telco
GROUP BY 
  Churn;

Churn,pocet_zakazniku,prumerna_mesicni_platba,prumerna_doba_u_firmy_v_mesicich
No,5174,61.27,37.6
Yes,1869,74.44,18.0


_Zde lze říct, že cena hraje určitě roli v rozhodování k odchodu._
_Klienti, kteří odcházejí platí v průměru 74,44 a ti věrní jen 61,27. Čili to znamená, že odcházejí ti co mají vlastně služby dražší._
_Když se podíváme na dobu jakou u nás jsou, tak jde vidět že vydrží kolem roka a půl. Možná bychom měli tento termín pohlídat a zkusit je kontaktovat s nabídkou výhodnějších služeb?_

# Analýza využívaných služeb a technické podpory

Zkoumám jaké konkrétní internetové služby zákazníci využívají a jestli má na jejich odchod vliv služba technické podpory. Chci ověřit, jestli drahé služby bez dostatečné péče nevedou k nespokojenosti a tudíž odchodu.

In [0]:
SELECT 
  CONCAT(InternetService, ' (', TechSupport, ')') AS sluzba_a_podpora,
  Churn,
  COUNT(*) AS pocet_zakazniku,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(PARTITION BY InternetService, TechSupport), 2) AS procento_v_ramci_kombinace
FROM 
  workspace.default.telco
GROUP BY 
  InternetService,
  TechSupport,
  Churn
ORDER BY 
  InternetService,
  TechSupport,
  Churn;

sluzba_a_podpora,Churn,pocet_zakazniku,procento_v_ramci_kombinace
DSL (No),No,898,72.24
DSL (No),Yes,345,27.76
DSL (Yes),No,1064,90.32
DSL (Yes),Yes,114,9.68
Fiber optic (No),No,1129,50.63
Fiber optic (No),Yes,1101,49.37
Fiber optic (Yes),No,670,77.37
Fiber optic (Yes),Yes,196,22.63
No (No internet service),No,1413,92.60
No (No internet service),Yes,113,7.40


Databricks visualization. Run in Databricks to view.

![Graf odchodovosti dle typu internetu a technické podpory](images/graf_odchodovost_s_tech_sup.png)

_Zde vidíme, že nejrizikovější skupinou jsou zákazníci s optickým internetem, kteří nemají aktivovanou technickou podporu. Podle tabulky v této skupině odchází 49,37 % klientů. Pokud však zákazníci s optikou technickou podporu mají, míra odchodu klesá na 22,63 %._

_U DSL internetu bez podpory odejde 27,76 % zákazníků a s podporou odejde 9,68 % zákazníků, což je podobné jako u té optiky._

_Čili z toho vyplývá doporučení, že optický internet je potřeba nabízet nejlépe rovnou s technickou podporou._

# Analýza věrnosti zákazníků 

Nyní si uděláme select podle doby, co jsou klienti u naší společnosti. Ověříme tak, jestli je kritické období na začátku smluv a nebo ztrácíme klienty, kteří jsou u nás delší dobu.

In [0]:
SELECT 
  CASE 
    WHEN tenure <= 6 THEN '01. 0-6 měsíců'
    WHEN tenure <= 12 THEN '02. 6-12 měsíců'
    WHEN tenure <= 24 THEN '03. 1-2 roky'
    ELSE '04. Více než 2 roky'
  END AS segment_vernosti,
  Churn,
  COUNT(*) AS pocet_zakazniku,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(PARTITION BY 
    CASE 
      WHEN tenure <= 6 THEN '01. 0-6 měsíců'
      WHEN tenure <= 12 THEN '02. 6-12 měsíců'
      WHEN tenure <= 24 THEN '03. 1-2 roky'
      ELSE '04. Více než 2 roky'
    END
  ), 2) AS procento_v_ramci_segmentu
FROM 
  workspace.default.telco
GROUP BY 
  1, 
  Churn
ORDER BY 
  segment_vernosti, 
  Churn;

segment_vernosti,Churn,pocet_zakazniku,procento_v_ramci_segmentu
01. 0-6 měsíců,No,697,47.06
01. 0-6 měsíců,Yes,784,52.94
02. 6-12 měsíců,No,452,64.11
02. 6-12 měsíců,Yes,253,35.89
03. 1-2 roky,No,730,71.29
03. 1-2 roky,Yes,294,28.71
04. Více než 2 roky,No,3295,85.96
04. Více než 2 roky,Yes,538,14.04


Databricks visualization. Run in Databricks to view.

![Graf věrnosti zákazníků](images/graf_vernosti.png)

_Zde vidíme, že obdobím pro odchod zákazníka je hned prvních 6 měsíců od začátku smlouvy. Odchází nám 52, 94% nově získaných zákazníků. Pak lze z tabulky vyčíst i to, že postupem času se to lepší. Ti, co tu jsou pak déle než 2 roky, tak míra odchodů klesá na 14, 04%._

_Daří se nám získávat nové klienty, ale nedokážeme je udržet. Bude tedy klíčové se na to zaměřit._




# Analýza vlivu platebních metod na odchod zákazníků
Zkoumám, zda má způsob platby vliv na loajalitu zákazníků. Předpokládám, že zákazníci, kteří musí platit účty manuálně, odcházejí častěji než ti, kteří mají nastavenou automatickou platbu z bankovního účtu.

In [0]:
SELECT 
  PaymentMethod,
  Churn,
  COUNT(*) AS pocet_zakazniku,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(PARTITION BY PaymentMethod), 2) AS procento_v_ramci_metody
FROM 
  workspace.default.telco
GROUP BY 
  PaymentMethod, 
  Churn
ORDER BY 
  PaymentMethod, 
  Churn;

PaymentMethod,Churn,pocet_zakazniku,procento_v_ramci_metody
Bank transfer (automatic),No,1286,83.29
Bank transfer (automatic),Yes,258,16.71
Credit card (automatic),No,1290,84.76
Credit card (automatic),Yes,232,15.24
Electronic check,No,1294,54.71
Electronic check,Yes,1071,45.29
Mailed check,No,1304,80.89
Mailed check,Yes,308,19.11


Databricks visualization. Run in Databricks to view.

![Graf odchodovosti podle platebních metod](images/graf_platebni_metody.png)

_Tak tady jasně vyplývá, že i volba platební metody má vliv na odchodovost zákazníků. U těch jednorázových plateb je odchodovost 45,29% a naopak u inkasních plateb se to pohybuje mezi 15 až 17%._

_Měli bychom zákazníky motivovat k tomu, aby si nastavovali inkasní způsoby plateb._


# Analýza vlivu streamovacích služeb na odchod zákazníků
Zkoumám jaký dopad má využívání doplňkových multimediálních služeb na loajalitu zákazníků. Chci zjistit, zda tyto služby fungují jako „kotva“, která zákazníka u firmy drží, nebo zda pro něj představují zbytnou finanční zátěž, která vede k odchodu.


In [0]:
SELECT 
  StreamingTV,
  StreamingMovies,
  Churn,
  COUNT(*) AS pocet_zakazniku,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(PARTITION BY StreamingTV, StreamingMovies), 2) AS procento_v_ramci_kombinace
FROM 
  workspace.default.telco
GROUP BY 
  StreamingTV, 
  StreamingMovies, 
  Churn
ORDER BY 
  StreamingTV, 
  StreamingMovies, 
  Churn;
  

StreamingTV,StreamingMovies,Churn,pocet_zakazniku,procento_v_ramci_kombinace
No,No,No,1323,65.56
No,No,Yes,695,34.44
No,Yes,No,545,68.81
No,Yes,Yes,247,31.19
No internet service,No internet service,No,1413,92.60
No internet service,No internet service,Yes,113,7.40
Yes,No,No,524,68.32
Yes,No,Yes,243,31.68
Yes,Yes,No,1369,70.57
Yes,Yes,Yes,571,29.43


_Z analýzy vyplývá, že doplňkové zábavní služby mají pozitivní vliv na retenci. Zákazníci, kteří nevyužívají ani jednu streamovací službu, odcházejí v 34,44 % případů._ 
_Pokud si však aktivují jak StreamingTV, tak StreamingMovies, tak míra odchodu klesá na 29,43 %._ 
_Zábavní obsah tedy funguje jako prvek, který zvyšuje loajalitu zákazníků vůči naší společnosti._
_Zároveň data ukazují, že specifickou a velmi stabilní skupinou jsou zákazníci bez internetových služeb (pouze hlasové služby), u kterých je míra odchodu pouhých 7,40 %._
_Doporučení tedy pro management je, že bychom měli podporovat prodej entertainment balíčků/služeb._ 
_Zákazníkům, kteří mají internet, ale dosud nestreamují, bychom měli nabídnout např. zkušební období pro StreamingTV a Movies zdarma. Jakmile si na tyto služby zvyknou, pravděpodobnost jejich odchodu se sníží._

# Vizualizace kombinovaného vlivu streamovacích služeb

Abychom mohli v grafu přehledně porovnat vliv obou streamovacích služeb najednou, spojujeme tyto dva textové sloupce pomocí funkce CONCAT do jedné společné metriky. To nám umožní sledovat efekt obou služeb na jedné ose a identifikovat, zda kombinace obou služeb funguje jako silnější retenční prvek.

In [0]:
SELECT 
  CONCAT('TV: ', StreamingTV, ' | Movie: ', StreamingMovies) AS kombinace_streamovani,
  Churn,
  COUNT(*) AS pocet_zakazniku,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(PARTITION BY StreamingTV, StreamingMovies), 2) AS procento_v_ramci_kombinace
FROM 
  workspace.default.telco
GROUP BY 
  StreamingTV, 
  StreamingMovies, 
  Churn
ORDER BY 
  StreamingTV, 
  StreamingMovies, 
  Churn;
  

kombinace_streamovani,Churn,pocet_zakazniku,procento_v_ramci_kombinace
TV: No | Movie: No,No,1323,65.56
TV: No | Movie: No,Yes,695,34.44
TV: No | Movie: Yes,No,545,68.81
TV: No | Movie: Yes,Yes,247,31.19
TV: No internet service | Movie: No internet service,No,1413,92.60
TV: No internet service | Movie: No internet service,Yes,113,7.40
TV: Yes | Movie: No,No,524,68.32
TV: Yes | Movie: No,Yes,243,31.68
TV: Yes | Movie: Yes,No,1369,70.57
TV: Yes | Movie: Yes,Yes,571,29.43


Databricks visualization. Run in Databricks to view.

![Graf odchodovosti podle streamovacích služeb](images/graf_streamovaci_sluzby.png)

# Analýza odchodovosti podle věkových skupin

Zkoumám jak se liší chování starších zákazníků oproti mladší klientské bázi. 
Z byznysového pohledu mají mladší klienti sice delší životní cyklus a vyšší potenciál pro nákup doplňkových digitálních služeb, ale je klíčové ověřit, zda nám mladší báze neutíká ke konkurenci rychleji a jak stabilní jsou naopak starší ročníky.
Tento dotaz nám tedy pomůže odhalit, zda věk hraje roli v loajalitě vůči firmě.

In [0]:
WITH procenta_churnu AS (
  SELECT 
    SeniorCitizen,
    Churn,
    COUNT(*) AS pocet_zakazniku,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(PARTITION BY SeniorCitizen), 2) AS procento_v_ramci_skupiny
  FROM 
    workspace.default.telco
  GROUP BY 
    SeniorCitizen, 
    Churn
)
SELECT 
  SeniorCitizen,
  procento_v_ramci_skupiny AS procento_odchodu
FROM 
  procenta_churnu
WHERE 
  Churn = 'Yes';

SeniorCitizen,procento_odchodu
0,23.61
1,41.68


Databricks visualization. Run in Databricks to view.

![Graf odchodovosti podle věku](images/graf_odchodovost_vek.png)

_Tady vyplývá, že starší zákazníci vykazují výrazně vyšší míru odchodu než zbytek zákazníků._ 
_Zatímco u mladších je odchodovost na úrovni 23,61 %, u seniorů stoupá na 41,68 %._
_Tento výsledek jednoznačně potvrzuje, že zaměření na mladší klientskou bázi je strategicky správný krok, protože tito klienti vykazují mnohem vyšší stabilitu a loajalitu vůči firmě. Naopak vysoký odchod seniorů naznačuje, že stávající nastavení služeb nebo procesů (např. digitální fakturace či chybějící asistence) pro ně představuje jakousi bariéru._
_Zde tedy při akvizicích se dobrý se soustředit na mladší segment. Zároveň ale je nutné stávající senior. zíkazníky stabilizovat (např. zjednodušením platebních metod), abychom zamezili zbytečnému odtoku dosavadních zisků._

# Hlubší analýza segmentu mladších zákazníků

Abychom dokázali mladou klientskou bázi efektivně rozvíjet, zkoumáme jejich chování v závislosti na typu internetového připojení a využívání bezpečnostních prvků. 
U mladší generace se předpokládá vysoká tech gramotnost, proto chci ověřit, zda absence zabezpečení nebo nevhodný typ internetu koreluje s jejich odchodem ke konkurenci.

In [0]:
SELECT 
  InternetService,
  OnlineSecurity,
  OnlineBackup,
  COUNT(*) AS pocet_mladych_zakazniku,
  SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS odeslo,
  ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS procento_odchodu_mladych
FROM 
  workspace.default.telco
WHERE 
  SeniorCitizen = 0
GROUP BY 
  InternetService,
  OnlineSecurity,
  OnlineBackup
ORDER BY 
  procento_odchodu_mladych DESC;

InternetService,OnlineSecurity,OnlineBackup,pocet_mladych_zakazniku,odeslo,procento_odchodu_mladych
Fiber optic,No,No,998,545,54.61
Fiber optic,No,Yes,615,222,36.10
DSL,No,No,698,229,32.81
Fiber optic,Yes,No,271,77,28.41
Fiber optic,Yes,Yes,381,60,15.75
DSL,No,Yes,379,58,15.30
DSL,Yes,No,507,58,11.44
No,No internet service,No internet service,1474,108,7.33
DSL,Yes,Yes,578,36,6.23


_Z dotazu jasně vyplývá, že samotný Rychlý optický internet u mladé generace k udržení nestačí a bude zapotřebí soustředit se na doplňkový služby._
_Zákazníci s optickým internet, co nemají online security ani online backup, tak mají míru odchodovosti 54,61%, což je nějakých 545 zákazníků z 998._
_Pokud na na optice mají obě služby, tak míra odchodu klesá viz tabulka na 15,75%._
_U DSLka je to trošku stabilnější, ale i zde ta kombinace obou bezpečnostních prvků snížuje odchodovost na 6,23%._
_Čili bychom jim neměli optiku prodávat samotnou, ale přifařit tam online security i backup a mohli bychom tak snížit odchodovst z 54,61% na 15,75%._


_Tady ještě upravuji dotaz pro vizualizaci, kategorickou osu X._

In [0]:
SELECT 
  CONCAT(InternetService, ' (Security: ', OnlineSecurity, ', Backup: ', OnlineBackup, ')') AS Kombinace_sluzeb,
  ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS procento_odchodu_mladych
FROM 
  workspace.default.telco
WHERE 
  SeniorCitizen = 0
GROUP BY 
  InternetService,
  OnlineSecurity,
  OnlineBackup
ORDER BY 
  procento_odchodu_mladych DESC;

Kombinace_sluzeb,procento_odchodu_mladych
"Fiber optic (Security: No, Backup: No)",54.61
"Fiber optic (Security: No, Backup: Yes)",36.10
"DSL (Security: No, Backup: No)",32.81
"Fiber optic (Security: Yes, Backup: No)",28.41
"Fiber optic (Security: Yes, Backup: Yes)",15.75
"DSL (Security: No, Backup: Yes)",15.30
"DSL (Security: Yes, Backup: No)",11.44
"No (Security: No internet service, Backup: No internet service)",7.33
"DSL (Security: Yes, Backup: Yes)",6.23


Databricks visualization. Run in Databricks to view.

![Graf služeb u mladých zákazníků](images/graf_sluzby_mladi.png)

# Analýza smluv a finančního chování mladých zákazníků

Abychom plně porozuměli motivacím mladší generace, zkoumám interakci mezi typem smluvního vztahu a zvolenou platební metodou.

In [0]:
SELECT 
  Contract,
  PaymentMethod,
  COUNT(*) AS pocet_mladych_zakazniku,
  SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS odeslo,
  ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS procento_odchodu_mladych
FROM 
  workspace.default.telco
WHERE 
  SeniorCitizen = 0
GROUP BY 
  Contract,
  PaymentMethod
ORDER BY 
  procento_odchodu_mladych DESC;

Contract,PaymentMethod,pocet_mladych_zakazniku,odeslo,procento_odchodu_mladych
Month-to-month,Electronic check,1337,690,51.61
Month-to-month,Bank transfer (automatic),477,155,32.49
Month-to-month,Credit card (automatic),433,129,29.79
Month-to-month,Mailed check,821,240,29.23
One year,Electronic check,284,51,17.96
One year,Bank transfer (automatic),326,34,10.43
One year,Credit card (automatic),350,31,8.86
Two year,Electronic check,150,13,8.67
One year,Mailed check,323,21,6.50
Two year,Bank transfer (automatic),508,16,3.15


_Tady přicházím na to, že způsob placení a délka závazku jsou pro mladou generaci klíčovými indikátory loajality._
_Mladí zákazníci s měsíčním tarifem, kteří platí manuálně elektronickým šekem, vykazují extrémní odchodovost 51,61 % (odešlo 690 z 1337 klientů). Čili ruční placení jim dává každý měsíc prostor k přehodnocení služeb._
_Z tabulky jde krásně vidět, že pokud na měsíčním tarifu je automatická platba, tak odchodovost klesá. U dvouletých smluv je to úplně minimální bez ohledu na způsob platby._
_Tady bysme měli aktivně tlačit automatizované platby a podařilo by se snížit odchodovost._





_Ještě upravuji SQL dotaz pro přehlednější graf._

In [0]:
SELECT 
  CONCAT(Contract, ' (Platba: ', PaymentMethod, ')') AS Smlouva_a_platba,
  ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS procento_odchodu_mladych
FROM 
  workspace.default.telco
WHERE 
  SeniorCitizen = 0
GROUP BY 
  Contract,
  PaymentMethod
ORDER BY 
  procento_odchodu_mladych DESC;

Smlouva_a_platba,procento_odchodu_mladych
Month-to-month (Platba: Electronic check),51.61
Month-to-month (Platba: Bank transfer (automatic)),32.49
Month-to-month (Platba: Credit card (automatic)),29.79
Month-to-month (Platba: Mailed check),29.23
One year (Platba: Electronic check),17.96
One year (Platba: Bank transfer (automatic)),10.43
One year (Platba: Credit card (automatic)),8.86
Two year (Platba: Electronic check),8.67
One year (Platba: Mailed check),6.50
Two year (Platba: Bank transfer (automatic)),3.15


Databricks visualization. Run in Databricks to view.

![Graf smluv a plateb u mladých](images/graf_smlouvy_platby_mladi.png)

# Analýza cyklu a finanční zátěže mladých zákazníků

V této fázi zkoumám, ve kterém momentě mladí zákazníci nejčastěji odcházejí a jakou roli v jejich rozhodování hraje výše měsíčního vyúčtování. 

In [0]:
SELECT 
  CASE 
    WHEN tenure <= 3 THEN '0-3 měsíce (Nováčci)'
    WHEN tenure <= 12 THEN '4-12 měsíců (Kritický 1. rok)'
    WHEN tenure <= 24 THEN '1-2 roky (Stabilizovaní)'
    ELSE 'Více než 2 roky (Věrní)'
  END AS Doba_u_firmy,
  COUNT(*) AS pocet_mladych_zakazniku,
  ROUND(AVG(MonthlyCharges), 2) AS prumerna_mesicni_platba,
  SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS odeslo,
  ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS procento_odchodu_mladych
FROM 
  workspace.default.telco
WHERE 
  SeniorCitizen = 0
GROUP BY 
  CASE 
    WHEN tenure <= 3 THEN '0-3 měsíce (Nováčci)'
    WHEN tenure <= 12 THEN '4-12 měsíců (Kritický 1. rok)'
    WHEN tenure <= 24 THEN '1-2 roky (Stabilizovaní)'
    ELSE 'Více než 2 roky (Věrní)'
  END
ORDER BY 
  procento_odchodu_mladych DESC;

Doba_u_firmy,pocet_mladych_zakazniku,prumerna_mesicni_platba,odeslo,procento_odchodu_mladych
0-3 měsíce (Nováčci),908,51.1,473,52.09
4-12 měsíců (Kritický 1. rok),958,56.15,347,36.22
1-2 roky (Stabilizovaní),856,58.13,211,24.65
Více než 2 roky (Věrní),3179,67.63,362,11.39


Databricks visualization. Run in Databricks to view.

_Tak tady největší problém nepředstavuje cena služeb, ale proces adaptace na začátku jejich cesty u nás._

_Dle tabulky nám odchází 52,09%, což je 473 zákazníků z 908, co vydrží sotva 3 měsíce. Čili to je bych řekla klíčové._

_I druhá skupina mezi čtyřmi a 12 měsíci je ale důležitá, jelikož nám tam odchází 36,22%, což není úplně malé číslo._

_Průměrná mědíční útrata stoupá z 51,1 u nováčků na 67,63 u zákazníků, kteří s námi zůstávají déle než 2 roky. Tito věrní klienti navíc vykazují minimální odchodovost a ta je 11,39 %._

_Problém tedy není v tom, že by služby byly drahé, protože vlastně ti nováčci platí v průměru nejméně. Problém bude asi v tom, že jim produkt buď hned od začátku nefunguje podle představ a nebo nezvládnou první krok?_

![Životní cyklus mladých zákazníků](images/graf_zivotni_cyklus_mladi.png)